In [1]:
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import sys
sys.path.append('C:/Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils')

from NN_utils import *
import torch
import torch.nn as nn
from torchvision import datasets, transforms
#from torchsummary import summary
import time
from types import SimpleNamespace
import pickle
import gc
from torch.utils.data import DataLoader


In [2]:

# --- Parameters ---
RNN_params = {
    "N_in": 784,
    "N_out": 10,
    "N_neurons": 50,
    "N_layers": 3,
}

#Create model
model = Oscillator_RNN_parallel_jit(params=RNN_params)
model.init_esn_weights(reservoir=True)
model.save_activations = True

# Set model to eval mode and move to device
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --- JIT Compile ---
scripted_model = torch.jit.script(model)
scripted_model.eval()  # Still set to eval
scripted_model.to(device)

# --- Load a batch of FashionMNIST ---
transform = transforms.Compose([transforms.ToTensor()])
dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
loader = DataLoader(dataset, batch_size=1000, shuffle=False)

# Get one batch
img, label = next(iter(loader))
x = img.to(device)

# --- Forward Pass with Timing ---
start_time = time.time()
with torch.no_grad():
    y_pred = scripted_model(x)
end_time = time.time()

print(f'Forward pass time with TorchScript: {end_time - start_time:.6f} seconds')


RuntimeError: 
Wrong type for attribute assignment. Expected NoneType but got Dict[str, List[Tensor]]:
  File "C:\Users/Admin/Desktop/PhD/simulation/simulation_python/nonlinear_oscillator_network/Utils\NN_utils.py", line 1558
        
        if save_activations:
            self.activations = {f"layer{l}": [] for l in range(self.N_layers)}
            ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~ <--- HERE
    
        batch_size = X.size(0)
